# Attention Event Prompt Lab

This notebook is a sandbox to iterate on event narrative prompts before deployment.

Goal: produce analyst-style causal explanations for:
- `What Happened`
- `Why It Happened`
- `Affected Assets`

and avoid stat-dump prose in the causal section.

## Setup

This uses the same LLM client and JSON schema as production (`services/llm.py`, `services/attention_agentic.py`).

If `llm_client` is `None`, set env vars first (for example in this notebook cell):

```python
import os
os.environ["LLM_PROVIDER"] = "openai"  # or "azure_openai"
os.environ["OPENAI_API_KEY"] = "..."
os.environ["LLM_MODEL"] = "gpt-4.1-mini"
```

This notebook includes a bootstrap cell that pulls API config directly from production pipeline jobs via Azure CLI, then calls the same `services.llm` resolver used in runtime.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "services").exists():
    for parent in [ROOT, *ROOT.parents]:
        if (parent / "services").exists():
            ROOT = parent
            break

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Repo root: {ROOT}")

Repo root: /mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code/Users/omai.r/spectral_nature/streamlit_alpaca_app


In [2]:
import json
import os
import subprocess

# Pull the same LLM-related env vars used by production pipeline jobs.
# This mirrors runtime config without hardcoding local API settings.
RESOURCE_GROUP = "sn-pipeline-rg-03130136"
CONFIG_SOURCE_KIND = "job"  # 'job' or 'app'
CONFIG_SOURCE_NAME = "news-ingest-and-features"
FORCE_REASONING_EFFORT = "high"  # set to "" to disable forced reasoning in this notebook

ALLOWED_ENV_KEYS = [
    "LLM_PROVIDER",
    "LLM_MODEL",
    "LLM_DEPLOYMENT",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_VERSION",
    "LLM_API_KEY_SECRET_NAME",
    "OPENAI_API_KEY_SECRET_NAME",
    "LLM_TEMPERATURE",
    "LLM_TIMEOUT_SECONDS",
    "LLM_REASONING_EFFORT",
    "AZURE_KEY_VAULT_NAME",
    "KEY_VAULT_NAME",
    "AZURE_CLIENT_ID",
]

def _fetch_env_from_azure(kind: str, name: str, resource_group: str) -> dict[str, str]:
    if kind == "job":
        cmd = [
            "az", "containerapp", "job", "show",
            "-n", name,
            "-g", resource_group,
            "--query", "properties.template.containers[0].env",
            "-o", "json",
        ]
    elif kind == "app":
        cmd = [
            "az", "containerapp", "show",
            "-n", name,
            "-g", resource_group,
            "--query", "properties.template.containers[0].env",
            "-o", "json",
        ]
    else:
        raise ValueError(f"Unsupported CONFIG_SOURCE_KIND: {kind}")

    raw = subprocess.check_output(cmd, text=True)
    rows = json.loads(raw)
    out: dict[str, str] = {}
    for row in rows:
        if not isinstance(row, dict):
            continue
        key = str(row.get("name") or "").strip()
        if not key:
            continue
        value = str(row.get("value") or "").strip()
        out[key] = value
    return out

def apply_production_llm_env(
    *,
    kind: str = CONFIG_SOURCE_KIND,
    name: str = CONFIG_SOURCE_NAME,
    resource_group: str = RESOURCE_GROUP,
    overwrite: bool = True,
) -> dict[str, str]:
    env_map = _fetch_env_from_azure(kind, name, resource_group)
    applied: dict[str, str] = {}
    for key in ALLOWED_ENV_KEYS:
        value = env_map.get(key, "")
        if not value:
            continue
        if overwrite or not os.getenv(key):
            os.environ[key] = value
            applied[key] = value

    if str(FORCE_REASONING_EFFORT or "").strip():
        os.environ["LLM_REASONING_EFFORT"] = str(FORCE_REASONING_EFFORT).strip().lower()
        applied["LLM_REASONING_EFFORT"] = os.environ["LLM_REASONING_EFFORT"]

    return applied

try:
    applied_env = apply_production_llm_env()
    print(f"Loaded production env from {CONFIG_SOURCE_KIND}:{CONFIG_SOURCE_NAME} ({RESOURCE_GROUP})")
    for k in sorted(applied_env):
        v = applied_env[k]
        if "SECRET" in k:
            v = "<secret-name>"
        print(f"  {k}={v}")
except Exception as exc:
    print("Could not auto-load production config from Azure.")
    print("Reason:", exc)
    print("If needed, run `az login --use-device-code` and rerun this cell.")

Loaded production env from job:news-ingest-and-features (sn-pipeline-rg-03130136)
  AZURE_CLIENT_ID=4f7c0e9a-2828-45cf-9153-86bbf52acf4e
  AZURE_KEY_VAULT_NAME=snpipelinekv03130136
  AZURE_OPENAI_ENDPOINT=https://omai-mn1g40ef-eastus2.cognitiveservices.azure.com/
  KEY_VAULT_NAME=snpipelinekv03130136
  LLM_API_KEY_SECRET_NAME=<secret-name>
  LLM_DEPLOYMENT=gpt-5.3-chat
  LLM_MODEL=gpt-5.3-chat
  LLM_PROVIDER=azure_openai
  LLM_REASONING_EFFORT=high
  LLM_TEMPERATURE=1


In [3]:
from services.llm import load_llm_client, load_llm_config
from services.attention_agentic import (
    EVENT_WRITER_SCHEMA,
    _fallback_event_writer,
    _has_causal_language,
    _looks_like_stat_dump,
    _text_overlap,
    _load_search_clients,
    _plan_candidate_research,
    _peer_candidates,
    _search_query_results,
    _candidate_context_documents,
    _documents_from_search_results,
    _chunk_source_documents,
    _extract_claims,
)

cfg = load_llm_config()
llm_client = load_llm_client()

print("LLM config loaded:", bool(cfg))
if cfg is not None:
    print({
        "provider": cfg.provider,
        "model": cfg.model,
        "base_url": cfg.base_url,
        "reasoning_effort": cfg.reasoning_effort,
    })
print("LLM client ready:", llm_client is not None)

LLM config loaded: True
{'provider': 'azure_openai', 'model': 'gpt-5.3-chat', 'base_url': 'https://omai-mn1g40ef-eastus2.cognitiveservices.azure.com/openai/v1', 'reasoning_effort': 'high'}
LLM client ready: True


## Event Input (Editable)

This is preloaded with your example. Edit `members`, `claims`, and `yield_facts` to test other days.

In [4]:
event_case: dict[str, Any] = {
    "event_title_seed": "Airlines lower, Energy higher",
    "members": [
        {"symbol": "USO", "sector": "Energy", "industry": "Commodities", "change_pct": 5.95, "candidate_score": 96.0},
        {"symbol": "BNO", "sector": "Energy", "industry": "Commodities", "change_pct": 4.35, "candidate_score": 90.0},
        {"symbol": "XOM", "sector": "Energy", "industry": "Integrated Oil & Gas", "change_pct": 3.36, "candidate_score": 89.0},
        {"symbol": "ABNB", "sector": "Consumer Discretionary", "industry": "Travel Services", "change_pct": -6.28, "candidate_score": 92.0},
        {"symbol": "LUV", "sector": "Industrials", "industry": "Airlines", "change_pct": -5.45, "candidate_score": 91.0},
        {"symbol": "UAL", "sector": "Industrials", "industry": "Airlines", "change_pct": -4.61, "candidate_score": 88.0},
        {"symbol": "BKNG", "sector": "Consumer Discretionary", "industry": "Travel Services", "change_pct": -3.68, "candidate_score": 84.0},
        {"symbol": "NIB", "sector": "Consumer Staples", "industry": "Soft Commodities", "change_pct": 22.61, "candidate_score": 75.0},
    ],
    "claims": [
        {
            "claim_text": "Crude prices rose on supply-disruption and shipping-risk headlines, raising expected jet fuel costs and squeezing airline margin assumptions.",
            "claim_type": "macro",
            "source": "Reuters",
            "is_same_day": True,
        },
        {
            "claim_text": "Travel names underperformed as investors priced weaker discretionary demand if tariff-related inflation stays firm and long-end rates remain elevated.",
            "claim_type": "macro",
            "source": "Bloomberg",
            "is_same_day": True,
        },
        {
            "claim_text": "Shutdown headlines raised concern about TSA staffing disruptions, adding operational risk to near-term travel volumes.",
            "claim_type": "policy",
            "source": "AP",
            "is_same_day": True,
        },
    ],
    "yield_facts": {
        "ust_2y": 3.88,
        "ust_10y": 4.44,
        "ust_30y": 4.98,
        "ust_2y_1d_bps": -8.0,
        "ust_10y_1d_bps": 2.0,
        "ust_30y_1d_bps": 5.0,
        "curve_2s10s": 0.56,
        "curve_2s10s_1d_bps": 10.0,
    },
}

pd.DataFrame(event_case["members"])

,symbol,sector,industry,change_pct,candidate_score
0,USO,Energy,Commodities,5.95,96.0
1,BNO,Energy,Commodities,4.35,90.0
2,XOM,Energy,Integrated Oil & Gas,3.36,89.0
3,ABNB,Consumer Discretionary,Travel Services,-6.28,92.0
4,LUV,Industrials,Airlines,-5.45,91.0
5,UAL,Industrials,Airlines,-4.61,88.0
6,BKNG,Consumer Discretionary,Travel Services,-3.68,84.0
7,NIB,Consumer Staples,Soft Commodities,22.61,75.0


In [5]:
BASELINE_SYSTEM_PROMPT = (
    "You write concise cross-asset market-event summaries. "
    "Use only the supplied facts and claims. Keep the surface summary at two sentences or less. "
    "Do not use canned oil/rates/risk phrases. "
    "Explain mechanism, not tape recap: connect the catalyst to transmission effects in pricing, margins, demand, funding, or risk appetite. "
    "Do not make why-happened text a long ticker/percent list. "
    "Affected-assets text should describe spillover and should not duplicate what-happened text. "
    "When numeric Treasury yield facts are supplied, use the actual bp moves."
)

PROMPT_VARIANTS: dict[str, str] = {
    "baseline_current": BASELINE_SYSTEM_PROMPT,
    "iteration_1_mechanism_first": (
        "You are a senior cross-asset strategist writing for PMs. "
        "Return concise JSON only. Use only supplied facts; do not invent facts. "
        "Critical rule for why_happened_text: lead with a causal chain in plain English before any numbers. "
        "Use this structure: catalyst -> transmission channel -> market pricing reaction. "
        "Transmission channels must be concrete (input costs, margins, volumes, funding costs, duration, policy/operations). "
        "Forbidden in why_happened_text: ticker lists, percentage rollups, yield recaps without mechanism, or repeating what_happened_text. "
        "If causality is uncertain, explicitly say what is uncertain and why."
    ),
    "iteration_2_hard_guardrails": (
        "You write institutional-quality event summaries. Output must be specific and mechanism-first. "
        "Use only supplied facts. Never fabricate a source or metric. "
        "For why_happened_text, include at least one explicit causal connector (because, after, amid, driven by, which pressured, which lifted). "
        "Start with the mechanism sentence, then optionally one supporting sentence with key numbers. "
        "Never open why_happened_text with Treasury/yield/ticker statistics. "
        "Do not list more than two tickers in why_happened_text. "
        "Affected-assets must focus on second-order spillover and cross-asset breadth, not a restatement of what_happened_text. "
        "If the evidence is mixed, state the conflict explicitly in one sentence."
    ),
}

def build_writer_user_prompt(case: dict[str, Any]) -> str:
    members_df = pd.DataFrame(case["members"])
    fallback = _fallback_event_writer(
        case["event_title_seed"],
        members_df,
        case.get("claims", []),
        yield_facts=case.get("yield_facts", {}),
    )
    payload = {
        "event_title_seed": case["event_title_seed"],
        "members": case["members"],
        "claims": case.get("claims", []),
        "yield_facts": case.get("yield_facts", {}),
        "fallback": fallback,
        "narrative_requirements": {
            "why_happened_text": "causal chain first, numbers second",
            "affected_assets_summary_text": "spillover summary without repeating what_happened_text",
        },
    }
    return json.dumps(payload, ensure_ascii=False, default=str)

def quality_checks(output: dict[str, Any]) -> dict[str, Any]:
    what_happened = str(output.get("what_happened_text") or "")
    why_happened = str(output.get("why_happened_text") or "")
    affected = str(output.get("affected_assets_summary_text") or "")
    surface = str(output.get("surface_summary") or "")
    return {
        "why_is_stat_dump": _looks_like_stat_dump(why_happened),
        "why_has_causal_language": _has_causal_language(why_happened),
        "surface_is_stat_dump": _looks_like_stat_dump(surface),
        "overlap_what_vs_affected": round(_text_overlap(what_happened, affected), 3),
        "overlap_surface_vs_why": round(_text_overlap(surface, why_happened), 3),
    }

def run_variant(case: dict[str, Any], variant_name: str, system_prompt: str) -> dict[str, Any]:
    if llm_client is None:
        raise RuntimeError("LLM client is not configured. Set env vars and rerun setup cells.")
    user_prompt = build_writer_user_prompt(case)
    output = llm_client.generate_json(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        schema_name="attention_event_writer",
        schema=EVENT_WRITER_SCHEMA,
    )
    checks = quality_checks(output)
    return {
        "variant": variant_name,
        "system_prompt": system_prompt,
        "output": output,
        "checks": checks,
    }

def render_result(result: dict[str, Any]) -> None:
    out = result["output"]
    chk = result["checks"]
    display(Markdown(f"### {result['variant']}"))
    display(Markdown("**What Happened**  \n" + str(out.get("what_happened_text") or "")))
    display(Markdown("**Why It Happened**  \n" + str(out.get("why_happened_text") or "")))
    display(Markdown("**Affected Assets**  \n" + str(out.get("affected_assets_summary_text") or "")))
    display(pd.DataFrame([chk]))

## RAG/Search Trace (Upstream)

This section runs the upstream pipeline that feeds the writer:

`queries -> search_results -> source_documents -> evidence_chunks -> claims -> writer_payload`

Use this to debug whether weak `Why It Happened` text is caused by poor retrieval/claims versus prompt writing.

In [6]:
def _event_members_frame(case: dict[str, Any]) -> pd.DataFrame:
    df = pd.DataFrame(case.get("members") or []).copy()
    if df.empty:
        raise ValueError("event_case['members'] is empty.")

    defaults: dict[str, Any] = {
        "symbol": "",
        "sector": "",
        "industry": "",
        "change_pct": 0.0,
        "candidate_score": 0.0,
        "macro_exposure_tags": [],
        "business_tags": [],
        "rates_role": "",
        "commodity_role": "",
        "asset_class": "equity",
        "security_type": "common_stock",
        "what_changed_text": "",
        "headline": "",
    }
    for col, default in defaults.items():
        if col not in df.columns:
            if isinstance(default, list):
                df[col] = [[] for _ in range(len(df))]
            else:
                df[col] = default

    df["symbol"] = df["symbol"].astype(str).str.upper().str.strip()
    df = df[df["symbol"].ne("")].copy()
    if df.empty:
        raise ValueError("event_case['members'] has no valid symbols.")

    df["change_pct"] = pd.to_numeric(df["change_pct"], errors="coerce").fillna(0.0)
    df["candidate_score"] = pd.to_numeric(df["candidate_score"], errors="coerce").fillna(0.0)
    df["abs_change_pct"] = df["change_pct"].abs()
    df["candidate_id"] = df["symbol"].map(lambda s: f"candidate::{s}")
    return df.reset_index(drop=True)


def _pick_candidate(case: dict[str, Any], symbol: str = "") -> tuple[dict[str, Any], pd.DataFrame]:
    members_df = _event_members_frame(case)
    selected_symbol = str(symbol or "").upper().strip()
    if selected_symbol:
        scoped = members_df[members_df["symbol"] == selected_symbol].head(1)
        if scoped.empty:
            raise ValueError(f"Symbol {selected_symbol} not found in event_case['members'].")
        row = scoped.iloc[0]
    else:
        row = members_df.sort_values(["candidate_score", "abs_change_pct"], ascending=[False, False]).iloc[0]

    candidate = row.to_dict()
    candidate.setdefault("asof_time_utc", pd.Timestamp.now(tz="UTC"))
    candidate.setdefault("bundle_id", f"symbol::{candidate['symbol']}")
    return candidate, members_df


def run_search_claim_trace(
    case: dict[str, Any],
    *,
    symbol: str = "",
    max_queries: int = 3,
) -> dict[str, Any]:
    if llm_client is None:
        raise RuntimeError("LLM client is not configured. Run setup cells first.")

    asof_time_utc = pd.Timestamp.now(tz="UTC")
    run_id = f"notebook-trace-{asof_time_utc.strftime('%Y%m%d%H%M%S')}"

    candidate, members_df = _pick_candidate(case, symbol=symbol)
    candidate["asof_time_utc"] = asof_time_utc

    peer_symbols = _peer_candidates(candidate, members_df, limit=5)
    plan = _plan_candidate_research(candidate, peer_symbols, llm_client)

    queries = [
        q for q in list(plan.get("queries") or [])
        if str((q or {}).get("query") or "").strip()
    ][: max(int(max_queries), 1)]

    serp_client, tavily_client = _load_search_clients()

    request_rows: list[dict[str, Any]] = []
    result_rows: list[dict[str, Any]] = []
    candidate_results: list[dict[str, Any]] = []

    per_query_budget = max(int(plan.get("evidence_budget") or 8) // max(len(queries), 1), 2)
    for query in queries:
        req_rows, res_rows = _search_query_results(
            str((query or {}).get("query") or "").strip(),
            candidate_id=str(candidate.get("candidate_id") or ""),
            run_id=run_id,
            asof_time_utc=asof_time_utc,
            serp_client=serp_client,
            tavily_client=tavily_client,
            budget=per_query_budget,
        )
        request_rows.extend(req_rows)
        result_rows.extend(res_rows)
        candidate_results.extend(res_rows)

    documents = _candidate_context_documents(
        candidate,
        news_payloads=None,
        context_payloads=None,
        filings_frame=None,
        fred_summary_frame=None,
        yield_curve_facts_frame=None,
        run_id=run_id,
        asof_time_utc=asof_time_utc,
        official_routes=[str(route or "") for route in list(plan.get("official_routes") or [])],
        priority_entities=[str(entity or "") for entity in list(plan.get("priority_entities") or []) if str(entity or "").strip()],
    )
    documents.extend(
        _documents_from_search_results(
            candidate,
            candidate_results,
            run_id=run_id,
            asof_time_utc=asof_time_utc,
        )
    )

    deduped_docs: list[dict[str, Any]] = []
    seen_doc_ids: set[str] = set()
    for item in documents:
        doc_id = str(item.get("document_id") or "").strip()
        if not doc_id or doc_id in seen_doc_ids:
            continue
        seen_doc_ids.add(doc_id)
        deduped_docs.append(item)

    chunks_df = _chunk_source_documents(
        deduped_docs,
        run_id=run_id,
        asof_time_utc=asof_time_utc,
        embedding_client=None,
    )

    claims = _extract_claims(
        candidate,
        chunks_df,
        run_id=run_id,
        asof_time_utc=asof_time_utc,
        hypotheses=list(plan.get("hypotheses") or []),
        llm_client=llm_client,
    )

    retained_claims = [item for item in claims if bool(item.get("is_same_day"))]
    writer_claims = (retained_claims[:8] if retained_claims else claims[:8])

    fallback = _fallback_event_writer(
        str(case.get("event_title_seed") or "Market move today"),
        members_df,
        writer_claims,
        yield_facts=case.get("yield_facts") or {},
    )

    writer_payload = {
        "event_title_seed": str(case.get("event_title_seed") or "Market move today"),
        "members": [
            {
                "symbol": str(row.get("symbol") or "").upper(),
                "sector": str(row.get("sector") or ""),
                "industry": str(row.get("industry") or ""),
                "change_pct": float(row.get("change_pct") or 0.0),
                "candidate_score": float(row.get("candidate_score") or 0.0),
            }
            for _, row in members_df.head(8).iterrows()
        ],
        "claims": [
            {
                "claim_text": item.get("claim_text"),
                "claim_type": item.get("claim_type"),
                "source": item.get("source"),
                "is_same_day": item.get("is_same_day"),
            }
            for item in writer_claims
        ],
        "yield_facts": case.get("yield_facts") or {},
        "fallback": fallback,
        "narrative_requirements": {
            "why_happened_text": "causal chain first, numbers second",
            "affected_assets_summary_text": "spillover summary without repeating what_happened_text",
        },
    }

    return {
        "asof_time_utc": asof_time_utc,
        "run_id": run_id,
        "candidate": candidate,
        "peer_symbols": peer_symbols,
        "plan": plan,
        "search_clients": {"serpapi": serp_client is not None, "tavily": tavily_client is not None},
        "requests_df": pd.DataFrame(request_rows),
        "results_df": pd.DataFrame(result_rows),
        "documents_df": pd.DataFrame(deduped_docs),
        "chunks_df": chunks_df,
        "claims_df": pd.DataFrame(claims),
        "writer_claims": writer_claims,
        "writer_payload": writer_payload,
    }

In [7]:
TRACE_SYMBOL = ""  # Optional: set to a symbol in event_case['members'], e.g. "ABNB"
TRACE_MAX_QUERIES = 3

trace = run_search_claim_trace(event_case, symbol=TRACE_SYMBOL, max_queries=TRACE_MAX_QUERIES)

print("run_id:", trace["run_id"])
print("candidate:", trace["candidate"].get("symbol"))
print("search clients:", trace["search_clients"])
print("query count:", len(trace["plan"].get("queries") or []))
print("requests/results/docs/chunks/claims:",
      len(trace["requests_df"]),
      len(trace["results_df"]),
      len(trace["documents_df"]),
      len(trace["chunks_df"]),
      len(trace["claims_df"]))

display(pd.DataFrame(trace["plan"].get("queries") or []))

display(trace["results_df"].head(12))
display(trace["documents_df"][[c for c in ["source_kind", "source_provider", "title", "url", "published_at"] if c in trace["documents_df"].columns]].head(12))
display(trace["chunks_df"][[c for c in ["chunk_id", "source_provider", "title", "chunk_text"] if c in trace["chunks_df"].columns]].head(12))
display(trace["claims_df"][[c for c in ["claim_text", "claim_type", "source", "is_same_day", "relevance_score", "causal_score", "confidence_score"] if c in trace["claims_df"].columns]].head(12))

run_id: notebook-trace-20260330044924
candidate: USO
search clients: {'serpapi': True, 'tavily': True}
query count: 4
requests/results/docs/chunks/claims: 6 9 9 6 6


,query,rationale
0,WTI crude price today reason oil rally,Identify whether crude benchmarks moved sharpl...
1,USO ETF news today United States Oil Fund,"Check for ETF-specific flows, roll activity, o..."
2,oil market news today OPEC supply disruption o...,Look for macro oil catalysts such as OPEC acti...
3,BNO XOM oil stocks move today why energy secto...,Confirm whether the move is sector-wide across...


,run_id,asof_time_utc,candidate_id,query_id,provider,result_id,title,url,snippet,source,published_at,authority_bucket,authority_rank
0,notebook-trace-20260330044924,2026-03-30 04:49:24.788510+00:00,candidate::USO,query::c9f0181df63cde4e,serpapi,query::c9f0181df63cde4e::serpapi::8d322e29d5af,Why are oil prices plunging? US oil prices cra...,https://m.economictimes.com/news/international...,,The Economic Times,"03/16/2026, 07:00 AM, +0000 UTC",web,3
1,notebook-trace-20260330044924,2026-03-30 04:49:24.788510+00:00,candidate::USO,query::c9f0181df63cde4e,serpapi,query::c9f0181df63cde4e::serpapi::c4df4631f86c,Crude Oil Price Outlook – Crude Continues to L...,https://www.fxempire.com/forecasts/article/cru...,,FXEmpire,"12/12/2025, 08:00 AM, +0000 UTC",web,3
2,notebook-trace-20260330044924,2026-03-30 04:49:24.788510+00:00,candidate::USO,query::c9f0181df63cde4e,tavily,query::c9f0181df63cde4e::tavily::error,,,"Tavily request failed status=432: {""detail"":{""...",tavily,,web,3
3,notebook-trace-20260330044924,2026-03-30 04:49:24.788510+00:00,candidate::USO,query::3594439fc249005f,serpapi,query::3594439fc249005f::serpapi::5913e77584df,How to Buy United States Oil Fund LP (USO),https://www.fool.com/investing/how-to-invest/e...,,The Motley Fool,"03/09/2026, 07:00 AM, +0000 UTC",press,2
4,notebook-trace-20260330044924,2026-03-30 04:49:24.788510+00:00,candidate::USO,query::3594439fc249005f,serpapi,query::3594439fc249005f::serpapi::3e6316890a87,"If Oil Prices Keep Climbing, These 3 ETFs Coul...",https://finance.yahoo.com/news/oil-prices-keep...,,Yahoo Finance,"03/16/2026, 07:00 AM, +0000 UTC",press,2
5,notebook-trace-20260330044924,2026-03-30 04:49:24.788510+00:00,candidate::USO,query::3594439fc249005f,tavily,query::3594439fc249005f::tavily::error,,,"Tavily request failed status=432: {""detail"":{""...",tavily,,web,3
6,notebook-trace-20260330044924,2026-03-30 04:49:24.788510+00:00,candidate::USO,query::d0ed0b6c156b3097,serpapi,query::d0ed0b6c156b3097::serpapi::200e7932748b,Oil Market Report - March 2026 – Analysis,https://www.iea.org/reports/oil-market-report-...,,IEA – International Energy Agency,"03/12/2026, 07:00 AM, +0000 UTC",web,3
7,notebook-trace-20260330044924,2026-03-30 04:49:24.788510+00:00,candidate::USO,query::d0ed0b6c156b3097,serpapi,query::d0ed0b6c156b3097::serpapi::8282e1d40839,Oil prices steady as Iran-US tensions and US d...,https://www.reuters.com/business/energy/oil-dr...,,reuters.com,"02/10/2026, 08:00 AM, +0000 UTC",wire,1
8,notebook-trace-20260330044924,2026-03-30 04:49:24.788510+00:00,candidate::USO,query::d0ed0b6c156b3097,tavily,query::d0ed0b6c156b3097::tavily::error,,,"Tavily request failed status=432: {""detail"":{""...",tavily,,web,3


,source_kind,source_provider,title,url,published_at
0,search,The Economic Times,Why are oil prices plunging? US oil prices cra...,https://m.economictimes.com/news/international...,NaT
1,search,FXEmpire,Crude Oil Price Outlook – Crude Continues to L...,https://www.fxempire.com/forecasts/article/cru...,NaT
2,search,tavily,,,NaT
3,search,The Motley Fool,How to Buy United States Oil Fund LP (USO),https://www.fool.com/investing/how-to-invest/e...,NaT
4,search,Yahoo Finance,"If Oil Prices Keep Climbing, These 3 ETFs Coul...",https://finance.yahoo.com/news/oil-prices-keep...,NaT
5,search,tavily,,,NaT
6,search,IEA – International Energy Agency,Oil Market Report - March 2026 – Analysis,https://www.iea.org/reports/oil-market-report-...,NaT
7,search,reuters.com,Oil prices steady as Iran-US tensions and US d...,https://www.reuters.com/business/energy/oil-dr...,NaT
8,search,tavily,,,NaT


,chunk_id,source_provider,title,chunk_text
0,doc::query::c9f0181df63cde4e::tavily::error::c...,tavily,,"Tavily request failed status=432: {""detail"":{""..."
1,doc::query::c9f0181df63cde4e::tavily::error::c...,tavily,,Please upgrade your plan or contact support@ta...
2,doc::query::3594439fc249005f::tavily::error::c...,tavily,,"Tavily request failed status=432: {""detail"":{""..."
3,doc::query::3594439fc249005f::tavily::error::c...,tavily,,Please upgrade your plan or contact support@ta...
4,doc::query::d0ed0b6c156b3097::tavily::error::c...,tavily,,"Tavily request failed status=432: {""detail"":{""..."
5,doc::query::d0ed0b6c156b3097::tavily::error::c...,tavily,,Please upgrade your plan or contact support@ta...


,claim_text,claim_type,source,is_same_day,relevance_score,causal_score,confidence_score
0,"Tavily request failed status=432: {""detail"":{""...",background,tavily,False,0.4,0.435,0.417
1,Please upgrade your plan or contact support@ta...,background,tavily,False,0.4,0.435,0.417
2,"Tavily request failed status=432: {""detail"":{""...",background,tavily,False,0.4,0.435,0.417
3,Please upgrade your plan or contact support@ta...,background,tavily,False,0.4,0.435,0.417
4,"Tavily request failed status=432: {""detail"":{""...",background,tavily,False,0.4,0.435,0.417
5,Please upgrade your plan or contact support@ta...,background,tavily,False,0.4,0.435,0.417


In [8]:
TRACE_PROMPT_VARIANT = "iteration_1_mechanism_first"

trace_writer_output = llm_client.generate_json(
    system_prompt=PROMPT_VARIANTS[TRACE_PROMPT_VARIANT],
    user_prompt=json.dumps(trace["writer_payload"], ensure_ascii=False, default=str),
    schema_name="attention_event_writer",
    schema=EVENT_WRITER_SCHEMA,
)

trace_writer_checks = quality_checks(trace_writer_output)

display(Markdown(f"### writer_from_trace ({TRACE_PROMPT_VARIANT})"))
display(Markdown("**What Happened**  \n" + str(trace_writer_output.get("what_happened_text") or "")))
display(Markdown("**Why It Happened**  \n" + str(trace_writer_output.get("why_happened_text") or "")))
display(Markdown("**Affected Assets**  \n" + str(trace_writer_output.get("affected_assets_summary_text") or "")))
display(pd.DataFrame([trace_writer_checks]))

print("writer input claim count:", len(trace["writer_payload"].get("claims") or []))
display(pd.DataFrame(trace["writer_payload"].get("claims") or []))

### writer_from_trace (iteration_1_mechanism_first)

**What Happened**  
Energy-related assets moved higher, led by oil-linked funds and integrated producers, while travel platforms and airline stocks declined. USO rose 5.95% and BNO gained 4.35%, with XOM up 3.36%. In contrast, ABNB fell 6.28%, LUV dropped 5.45%, UAL declined 4.61%, and BKNG lost 3.68%.

**Why It Happened**  
A rise in crude prices -> higher jet fuel and operating input costs for airlines while improving revenue expectations for oil producers -> markets repriced margins across the value chain. Airlines are highly sensitive to fuel costs, so higher crude feeds directly into operating expenses and compresses profitability assumptions, while upstream energy producers benefit from stronger realized prices and improved cash flow expectations. This margin redistribution pushed energy-linked assets higher while travel and airline equities sold off.

**Affected Assets**  
The move concentrated in the fuel supply chain: oil-linked ETFs and integrated producers outperformed while fuel‑sensitive transportation and travel platforms lagged, with airlines showing the most direct downside reaction.

,why_is_stat_dump,why_has_causal_language,surface_is_stat_dump,overlap_what_vs_affected,overlap_surface_vs_why
0,False,True,False,0.184,0.197


writer input claim count: 6


,claim_text,claim_type,source,is_same_day
0,"Tavily request failed status=432: {""detail"":{""...",background,tavily,False
1,Please upgrade your plan or contact support@ta...,background,tavily,False
2,"Tavily request failed status=432: {""detail"":{""...",background,tavily,False
3,Please upgrade your plan or contact support@ta...,background,tavily,False
4,"Tavily request failed status=432: {""detail"":{""...",background,tavily,False
5,Please upgrade your plan or contact support@ta...,background,tavily,False


## Run All Prompt Iterations

In [9]:
results: list[dict[str, Any]] = []
for name, prompt in PROMPT_VARIANTS.items():
    result = run_variant(event_case, name, prompt)
    results.append(result)

summary_rows = []
for r in results:
    out = r["output"]
    summary_rows.append(
        {
            "variant": r["variant"],
            "why_happened_text": out.get("why_happened_text", ""),
            **r["checks"],
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

KeyboardInterrupt: 

In [ ]:
for r in results:
    render_result(r)

### baseline_current

**What Happened**  
Energy-linked assets rallied while airlines and travel platforms sold off, with crude ETFs and integrated oil companies advancing and carriers and booking platforms lagging.

**Why It Happened**  
Supply‑disruption and shipping‑risk headlines pushed crude prices higher, lifting the expected cost of jet fuel and forcing investors to reprice airline margin assumptions. At the same time, concerns that tariff‑driven inflation and a higher long-end rate backdrop could weigh on discretionary travel demand—alongside shutdown headlines raising the risk of TSA staffing disruptions—added pressure to travel-exposed names.

**Affected Assets**  
The move funneled flows into crude-linked ETFs and large oil producers while travel-sensitive equities—from airlines to online booking platforms—saw selling as investors rotated away from fuel‑intensive and discretionary-demand exposures.

,why_is_stat_dump,why_has_causal_language,surface_is_stat_dump,overlap_what_vs_affected,overlap_surface_vs_why
0,False,True,False,0.238,0.23


...

### iteration_2_hard_guardrails

**What Happened**  
Energy-linked securities advanced sharply, with crude proxies and integrated oil names rising, while airlines and online travel platforms fell, including notable declines in ABNB, LUV, UAL, and BKNG.

**Why It Happened**  
Airline and travel shares weakened because crude prices jumped on supply‑disruption and shipping‑risk headlines, which lifted expected jet fuel costs and pressured airline margin assumptions. At the same time, investors priced softer discretionary travel demand amid tariff‑related inflation concerns and potential TSA staffing disruptions tied to shutdown headlines.

**Affected Assets**  
The move rippled beyond airlines into the broader travel complex while energy exposures bid higher across commodity funds and oil majors. Soft‑commodity exposure also saw an outsized gain, with NIB surging over 22%, highlighting broader commodity strength even as consumer travel demand proxies lagged.

,why_is_stat_dump,why_has_causal_language,surface_is_stat_dump,overlap_what_vs_affected,overlap_surface_vs_why
0,False,True,False,0.145,0.07


## Your Custom Prompt Sandbox

Edit the `CUSTOM_SYSTEM_PROMPT` text and rerun this cell.

In [ ]:
CUSTOM_SYSTEM_PROMPT = PROMPT_VARIANTS["iteration_1_mechanism_first"]

custom_result = run_variant(event_case, "custom_editable", CUSTOM_SYSTEM_PROMPT)
render_result(custom_result)

### custom_editable

**What Happened**  
Energy commodities and oil majors moved higher while airlines and travel services declined. USO rose 5.95%, BNO gained 4.35%, and XOM climbed 3.36%, while travel-related names including ABNB (-6.28%), LUV (-5.45%), UAL (-4.61%), and BKNG (-3.68%) underperformed.

**Why It Happened**  
Airline and travel stocks fell while energy names rallied because crude prices jumped after supply‑disruption and shipping‑risk headlines, which lifted expected jet fuel costs and pressured airline margin assumptions. The move was reinforced by investor concern about discretionary travel demand and possible operational disruptions from shutdown-related TSA staffing risks, though those demand concerns contrasted with the more direct cost pressure from rising oil.

**Affected Assets**  
The move showed cross‑asset spillover from energy into transport and consumer travel exposure: oil‑linked ETFs and integrated producers advanced while airlines and online travel agencies lagged as higher fuel inputs fed through to travel cost expectations and margin sensitivity across the sector.

,why_is_stat_dump,why_has_causal_language,surface_is_stat_dump,overlap_what_vs_affected,overlap_surface_vs_why
0,False,True,False,0.127,0.172


## Notes For Production Patch

When you pick a winner:
1. Copy that prompt into the event writer system prompt in `services/attention_agentic.py` (`_write_event_bundle`).
2. Keep current post-generation guards (`_looks_like_stat_dump`, overlap checks) enabled as a second layer.
3. Re-run this notebook against at least 3 event types before deploy.